# ExplicitStimulusWhiskerData

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `data`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/ExplicitStimulusWhiskerData.md`


Notebook source link: [ExplicitStimulusWhiskerData.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/ExplicitStimulusWhiskerData.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "ExplicitStimulusWhiskerData"
FAMILY = "data"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"ExplicitStimulusWhiskerData: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"ExplicitStimulusWhiskerData: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"ExplicitStimulusWhiskerData: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"ExplicitStimulusWhiskerData: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# ExplicitStimulusWhiskerData: stimulus-locked spiking with binomial GLM fit.
dt = 0.001
time = np.arange(0.0, 4.0, dt)
n_trials = 12

# Whisker-like drive: low-frequency envelope + punctate transients.
envelope = 0.8 * np.sin(2.0 * np.pi * 1.2 * time)
transients = np.zeros_like(time)
for center in [0.7, 1.5, 2.3, 3.2]:
    transients += np.exp(-0.5 * ((time - center) / 0.035) ** 2)
stimulus = envelope + 1.1 * transients
stimulus = (stimulus - np.mean(stimulus)) / np.std(stimulus)

spike_mat = np.zeros((n_trials, time.size), dtype=float)
for k in range(n_trials):
    trial_gain = 0.85 + 0.3 * rng.random()
    eta = -3.2 + trial_gain * (1.0 * stimulus)
    p = 1.0 / (1.0 + np.exp(-eta))
    spike_mat[k] = rng.binomial(1, p)

spike_prob = np.mean(spike_mat, axis=0)
X = np.column_stack([np.ones(time.size), stimulus])
fit = Analysis.fit_glm(X=X[:, 1:], y=spike_mat[0], fit_type="binomial", dt=1.0)
pred_prob = fit.predict(X[:, 1:])

fig, axes = plt.subplots(3, 1, figsize=(9.5, 7.2), sharex=False)
axes[0].plot(time, stimulus, color="k", linewidth=1.0)
axes[0].set_title(f"{TOPIC}: explicit stimulus")
axes[0].set_ylabel("z-score")

for k in range(min(10, n_trials)):
    t_spk = time[spike_mat[k] > 0]
    axes[1].vlines(t_spk, k + 0.6, k + 1.4, linewidth=0.4)
axes[1].set_ylabel("trial")
axes[1].set_title("Spike raster")

axes[2].plot(time, spike_prob, color="tab:blue", linewidth=1.0, label="trial mean")
axes[2].plot(time, pred_prob, color="tab:red", linewidth=1.0, label="binomial fit (trial 1)")
axes[2].set_title("Observed and fitted spike probability")
axes[2].set_xlabel("time [s]")
axes[2].set_ylabel("p(spike)")
axes[2].legend(loc="upper right")
plt.tight_layout()
plt.show()

fit_rmse = float(np.sqrt(np.mean((pred_prob - spike_mat[0]) ** 2)))
assert 0.9 < float(np.std(stimulus)) < 1.1
assert fit_rmse < 0.6
CHECKPOINT_METRICS = {
    "stimulus_std": float(np.std(stimulus)),
    "fit_rmse": float(fit_rmse),
}
CHECKPOINT_LIMITS = {
    "stimulus_std": (0.9, 1.1),
    "fit_rmse": (0.0, 0.6),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
